# Modélisation

Ce notebook démarre à partir de `dataset_consolide.csv` produit par `preparation.ipynb`. Il prend en charge le split, le preprocessing apprenant, l'entraînement, la comparaison des modèles et la sauvegarde du meilleur pipeline.


## Périmètre du notebook

Objectifs visés dans ce notebook :

- charger le dataset consolidé préparé en amont ;
- définir la cible métier et les variables explicatives ;
- séparer les jeux train et test ;
- construire un preprocessing apprenant reproductible ;
- comparer plusieurs pipelines de régression ;
- tester une ACP de modélisation uniquement à l'intérieur d'une `Pipeline` ;
- suivre les expériences dans MLflow lorsque la librairie est disponible ;
- sauvegarder le meilleur pipeline entraîné.


## Imports et configuration générale

Les imports ci-dessous couvrent uniquement la phase de modélisation.


In [1]:
from contextlib import nullcontext
from pathlib import Path
import json
import shutil
import sqlite3

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import mlflow
import mlflow.sklearn
from IPython.display import display
from scripts.project_config import DEFAULT_CONFIG_PATH, load_preparation_config

pd.set_option("display.max_columns", None)
SEED = 42


## Configuration

Le notebook ne dépend plus des fichiers sources bruts : il attend uniquement le dataset consolidé préparé en amont. Les chemins partagés sont chargés depuis `config/project_paths.yaml`.


In [ ]:
preparation_config = load_preparation_config(ensure_dirs=True)
DATASET_PATH = preparation_config["DATASET_PATH"]
ARTIFACTS_DIR = preparation_config["ARTIFACTS_DIR"]
MODEL_ARTIFACTS_DIR = ARTIFACTS_DIR / "models"
MODEL_COMPARISON_PATH = ARTIFACTS_DIR / "model_comparison.csv"
BEST_MODEL_PATH = MODEL_ARTIFACTS_DIR / "best_pipeline.joblib"
MLFLOW_DB_PATH = ARTIFACTS_DIR / "mlflow.db"
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.resolve()}"
MLFLOW_EXPERIMENT_NAME = "ocr_projet12_modelisation"
MLFLOW_ARTIFACTS_DIR = ARTIFACTS_DIR / "mlruns"
MLFLOW_EXPERIMENT_ARTIFACT_DIR = MLFLOW_ARTIFACTS_DIR / MLFLOW_EXPERIMENT_NAME
TARGET_COL = "target_yield_t_ha"

ARTIFACTS_DIR.mkdir(exist_ok=True)
MODEL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
MLFLOW_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_EXPERIMENT_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration chargée depuis : {DEFAULT_CONFIG_PATH.resolve()}")
print(f"Dataset consolidé : {DATASET_PATH}")
print(f"Dossier artefacts : {ARTIFACTS_DIR}")
print(f"Pipeline final : {BEST_MODEL_PATH}")
print(f"Comparaison modèles : {MODEL_COMPARISON_PATH}")
print(f"Base MLflow : {MLFLOW_DB_PATH}")
print(f"Tracking URI : {MLFLOW_TRACKING_URI}")
print(f"Artefacts MLflow : {MLFLOW_ARTIFACTS_DIR}")
print(f"Racine artefacts experiment : {MLFLOW_EXPERIMENT_ARTIFACT_DIR}")


## Chargement du dataset consolidé

`dataset_consolide.csv` constitue l'entrée unique de la modélisation.


In [3]:
model_data = pd.read_csv(DATASET_PATH)

dataset_overview = pd.DataFrame(
    {
        'indicateur': ['nb_lignes', 'nb_colonnes', 'valeurs_manquantes', 'doublons_lignes'],
        'valeur': [
            int(model_data.shape[0]),
            int(model_data.shape[1]),
            int(model_data.isna().sum().sum()),
            int(model_data.duplicated().sum()),
        ],
    }
)

display(dataset_overview)
display(model_data.head())
print(f"Dataset chargé : {model_data.shape[0]:,} lignes x {model_data.shape[1]} colonnes")


,indicateur,valeur
0,nb_lignes,29367
1,nb_colonnes,7
2,valeurs_manquantes,25779
3,doublons_lignes,0


,area,crop,year,target_yield_t_ha,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,Afghanistan,Maize,1990,1.7582,327.0,NaN,15.45
1,Afghanistan,Maize,1991,1.6800,327.0,NaN,14.57
2,Afghanistan,Maize,1992,1.5000,327.0,NaN,14.35
3,Afghanistan,Maize,1993,1.6786,327.0,NaN,14.96
4,Afghanistan,Maize,1994,1.6667,327.0,NaN,14.94


Dataset chargé : 29,367 lignes x 7 colonnes


## Cible et variables explicatives

La cible métier reste fixée à `target_yield_t_ha`.


In [4]:
target_col = TARGET_COL
feature_cols = [col for col in model_data.columns if col != target_col]

print(f"Cible métier retenue : {target_col}")
print(f"Variables explicatives disponibles : {feature_cols}")


Cible métier retenue : target_yield_t_ha
Variables explicatives disponibles : ['area', 'crop', 'year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']


## Séparation train / test et typologie des variables

Le split est réalisé avant tout preprocessing apprenant afin d'éviter les fuites de données.


In [5]:
X = model_data[feature_cols].copy()
y = model_data[target_col].copy()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [col for col in X.columns if col not in numeric_features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
)

split_summary = pd.DataFrame(
    {
        'jeu': ['train', 'test'],
        'nb_lignes': [X_train.shape[0], X_test.shape[0]],
        'nb_colonnes': [X_train.shape[1], X_test.shape[1]],
    }
)

display(split_summary)
print(f"Variables numériques : {numeric_features}")
print(f"Variables catégorielles : {categorical_features}")


,jeu,nb_lignes,nb_colonnes
0,train,23493,6
1,test,5874,6


Variables numériques : ['year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']
Variables catégorielles : ['area', 'crop']


## Preprocessing de modélisation et pipelines candidats

L'imputation, le scaling, l'encodage et l'éventuelle ACP de modélisation sont définis ici, après le split.


In [6]:
def make_numeric_transformer(use_pca=False):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
    if use_pca:
        steps.append(("pca", PCA()))
    return Pipeline(steps=steps)


def make_categorical_transformer():
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", encoder),
        ]
    )


def build_preprocessor(use_numeric_pca=False):
    return ColumnTransformer(
        transformers=[
            ("num", make_numeric_transformer(use_pca=use_numeric_pca), numeric_features),
            ("cat", make_categorical_transformer(), categorical_features),
        ]
    )


def build_pipeline(regressor, use_numeric_pca=False):
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(use_numeric_pca=use_numeric_pca)),
            ("regressor", regressor),
        ]
    )


numeric_transformer = make_numeric_transformer(use_pca=False)
categorical_transformer = make_categorical_transformer()
preprocessor = build_preprocessor(use_numeric_pca=False)

if len(numeric_features) >= 2:
    numeric_transformer_with_pca = make_numeric_transformer(use_pca=True)
    preprocessor_with_pca = build_preprocessor(use_numeric_pca=True)
else:
    numeric_transformer_with_pca = None
    preprocessor_with_pca = None

candidate_configs = {
    "dummy": {
        "pipeline": build_pipeline(DummyRegressor(strategy="mean")),
        "search_type": "none",
        "params": {},
    },
    "linear_regression": {
        "pipeline": build_pipeline(LinearRegression()),
        "search_type": "none",
        "params": {},
    },
    "ridge": {
        "pipeline": build_pipeline(Ridge(random_state=SEED)),
        "search_type": "grid",
        "params": {
            "regressor__alpha": [0.1, 1.0, 10.0, 100.0],
        },
    },
    "random_forest": {
        "pipeline": build_pipeline(RandomForestRegressor(random_state=SEED)),
        "search_type": "random",
        "params": {
            "regressor__n_estimators": [100, 200],
            "regressor__max_depth": [None, 20],
            "regressor__min_samples_leaf": [1, 2],
        },
        "n_iter": 4,
        "cv": 2,
    },
}

if len(numeric_features) >= 2:
    candidate_configs["ridge_numeric_pca"] = {
        "pipeline": build_pipeline(Ridge(random_state=SEED), use_numeric_pca=True),
        "search_type": "grid",
        "params": {
            "preprocessor__num__pca__n_components": [2, min(3, len(numeric_features)), len(numeric_features)],
            "regressor__alpha": [0.1, 1.0, 10.0],
        },
    }

candidate_overview = pd.DataFrame(
    [
        {
            "model": model_name,
            "search_type": config["search_type"],
            "cv_folds": config.get("cv", 3) if config["search_type"] in {"grid", "random"} else np.nan,
            "n_iter": config.get("n_iter", np.nan),
            "uses_numeric_pca": "pca" in model_name,
        }
        for model_name, config in candidate_configs.items()
    ]
)

display(candidate_overview)


,model,search_type,cv_folds,n_iter,uses_numeric_pca
0,dummy,none,NaN,NaN,False
1,linear_regression,none,NaN,NaN,False
2,ridge,grid,3.0,NaN,False
3,random_forest,random,2.0,4.0,False
4,ridge_numeric_pca,grid,3.0,NaN,True


## Entraînement, optimisation et MLflow

MLflow reste optionnel. Lorsqu'il est installé, un run est créé par modèle candidat et les métriques de test sont journalisées.


In [7]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
while mlflow.active_run() is not None:
    mlflow.end_run()

tracking_db_path = Path(MLFLOW_TRACKING_URI.removeprefix("sqlite:///"))
experiment_artifact_uri = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve().as_uri()

connection = sqlite3.connect(tracking_db_path)
cursor = connection.cursor()
cursor.execute(
    "SELECT experiment_id, name, artifact_location FROM experiments WHERE name = ?",
    (MLFLOW_EXPERIMENT_NAME,),
)
existing_row = cursor.fetchone()

if existing_row is not None:
    experiment_id, _, current_artifact_location = existing_row
    current_artifact_dir = Path(str(current_artifact_location).removeprefix("file://")).resolve()
    target_artifact_dir = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve()
    if current_artifact_dir.exists() and current_artifact_dir != target_artifact_dir:
        for child in current_artifact_dir.iterdir():
            destination = target_artifact_dir / child.name
            if not destination.exists():
                shutil.move(str(child), str(destination))
        if current_artifact_dir.exists() and current_artifact_dir.is_dir() and not any(current_artifact_dir.iterdir()):
            current_artifact_dir.rmdir()

    cursor.execute(
        "UPDATE experiments SET artifact_location = ? WHERE experiment_id = ?",
        (str(target_artifact_dir), experiment_id),
    )
    cursor.execute(
        """
        UPDATE runs
        SET artifact_uri = REPLACE(artifact_uri, ?, ?)
        WHERE experiment_id = ? AND artifact_uri LIKE ?
        """,
        (
            str(current_artifact_dir),
            str(target_artifact_dir),
            experiment_id,
            f"{current_artifact_dir}%",
        ),
    )
    connection.commit()
connection.close()

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    client.create_experiment(
        MLFLOW_EXPERIMENT_NAME,
        artifact_location=experiment_artifact_uri,
    )

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.sklearn.autolog(log_models=False, silent=True)

results = []
best_model_name = None
best_pipeline = None
best_rmse = np.inf
best_run_id = None

for model_name, config in candidate_configs.items():
    run_ctx = mlflow.start_run(run_name=model_name)

    with run_ctx:
        try:
            search_type = config["search_type"]
            estimator = config["pipeline"]
            cv_folds = config.get("cv", 3)

            if search_type == "grid":
                search = GridSearchCV(
                    estimator=estimator,
                    param_grid=config["params"],
                    scoring="neg_mean_squared_error",
                    cv=cv_folds,
                    n_jobs=-1,
                    error_score="raise",
                )
                search.fit(X_train, y_train)
                fitted_estimator = search.best_estimator_
                best_params = search.best_params_
                cv_rmse = float(np.sqrt(-search.best_score_))
            elif search_type == "random":
                search = RandomizedSearchCV(
                    estimator=estimator,
                    param_distributions=config["params"],
                    n_iter=config.get("n_iter", 4),
                    scoring="neg_mean_squared_error",
                    cv=cv_folds,
                    random_state=SEED,
                    n_jobs=-1,
                    error_score="raise",
                )
                search.fit(X_train, y_train)
                fitted_estimator = search.best_estimator_
                best_params = search.best_params_
                cv_rmse = float(np.sqrt(-search.best_score_))
            else:
                fitted_estimator = estimator.fit(X_train, y_train)
                best_params = {}
                cv_rmse = np.nan

            y_pred = fitted_estimator.predict(X_test)
            mae = float(mean_absolute_error(y_test, y_pred))
            rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            r2 = float(r2_score(y_test, y_pred))
            run_id = mlflow.active_run().info.run_id

            results.append(
                {
                    "model": model_name,
                    "search_type": search_type,
                    "cv_rmse": cv_rmse,
                    "test_mae": mae,
                    "test_rmse": rmse,
                    "test_r2": r2,
                    "best_params": json.dumps(best_params, ensure_ascii=True, default=str),
                    "status": "ok",
                    "run_id": run_id,
                }
            )

            mlflow.log_param("model_name", model_name)
            mlflow.log_param("search_type", search_type)
            for param_name, param_value in best_params.items():
                mlflow.log_param(param_name, param_value)
            mlflow.log_metric("test_mae", mae)
            mlflow.log_metric("test_rmse", rmse)
            mlflow.log_metric("test_r2", r2)
            if not np.isnan(cv_rmse):
                mlflow.log_metric("cv_rmse", cv_rmse)
            mlflow.sklearn.log_model(fitted_estimator, name="model")

            if rmse < best_rmse:
                best_model_name = model_name
                best_pipeline = fitted_estimator
                best_rmse = rmse
                best_run_id = run_id

            print(f"{model_name}: RMSE test = {rmse:.4f}, R2 test = {r2:.4f}")

        except Exception as exc:
            error_message = str(exc)
            results.append(
                {
                    "model": model_name,
                    "search_type": config["search_type"],
                    "cv_rmse": np.nan,
                    "test_mae": np.nan,
                    "test_rmse": np.nan,
                    "test_r2": np.nan,
                    "best_params": json.dumps({}, ensure_ascii=True),
                    "status": f"error: {error_message}",
                    "run_id": mlflow.active_run().info.run_id,
                }
            )
            mlflow.log_param("model_name", model_name)
            mlflow.log_param("status", "failed")
            mlflow.log_param("error_message", error_message[:250])
            print(f"{model_name} failed: {error_message}")

results_df = pd.DataFrame(results)
results_df["success"] = results_df["status"].eq("ok")
results_df = results_df.sort_values(["success", "test_rmse"], ascending=[False, True], na_position="last")
results_df = results_df.drop(columns=["success"]).reset_index(drop=True)

display(results_df)


2026/04/09 13:55:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


dummy: RMSE test = 7.4261, R2 test = -0.0004


2026/04/09 13:55:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


linear_regression: RMSE test = 4.1641, R2 test = 0.6855


2026/04/09 13:56:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/04/09 13:57:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


random_forest: RMSE test = 1.4540, R2 test = 0.9617


2026/04/09 13:58:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ridge_numeric_pca: RMSE test = 4.1640, R2 test = 0.6855


,model,search_type,cv_rmse,test_mae,test_rmse,test_r2,best_params,status,run_id
0,random_forest,random,1.924950,0.660507,1.453992,0.961650,"{""regressor__n_estimators"": 200, ""regressor__m...",ok,af98d31590a14e7aba815fa57b8b61a7
1,ridge,grid,4.189356,2.876631,4.164050,0.685464,"{""regressor__alpha"": 0.1}",ok,d857bb6e0378421b86a190b48ffbf034
2,ridge_numeric_pca,grid,4.189356,2.876631,4.164050,0.685464,"{""preprocessor__num__pca__n_components"": 4, ""r...",ok,571f23184c3c4d23bc78a7331771ad5c
3,linear_regression,none,NaN,2.877311,4.164111,0.685454,{},ok,7d129d6191034ad485a20c9b99ae6abf
4,dummy,none,NaN,5.453748,7.426054,-0.000357,{},ok,073ecd107ecd49858b585c2113241f62


## Sauvegarde des artefacts

Le meilleur pipeline et le tableau de comparaison final sont sauvegardés dans `artifacts/`.


In [16]:
results_df.to_csv(MODEL_COMPARISON_PATH, index=False)
joblib.dump(best_pipeline, BEST_MODEL_PATH)

best_model_summary = pd.DataFrame(
    {
        "indicateur": ["best_model_name", "best_test_rmse", "best_model_path", "comparison_path"],
        "valeur": [
            best_model_name,
            round(float(best_rmse), 4),
            str(BEST_MODEL_PATH),
            str(MODEL_COMPARISON_PATH),
        ],
    }
)

display(best_model_summary)
print(f"Meilleur pipeline sauvegardé : {BEST_MODEL_PATH.resolve()}")
print(f"Tableau de comparaison sauvegardé : {MODEL_COMPARISON_PATH.resolve()}")

with mlflow.start_run(run_name=f"best_model_summary::{best_model_name}"):
    mlflow.log_param("best_model_name", best_model_name)
    mlflow.log_metric("best_test_rmse", float(best_rmse))
    mlflow.log_param("source_best_run_id", best_run_id)
    mlflow.log_artifact(str(MODEL_COMPARISON_PATH))
    mlflow.log_artifact(str(BEST_MODEL_PATH))


,indicateur,valeur
0,best_model_name,random_forest
1,best_test_rmse,1.454
2,best_model_path,artifacts/models/best_pipeline.joblib
3,comparison_path,artifacts/model_comparison.csv


Meilleur pipeline sauvegardé : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/models/best_pipeline.joblib
Tableau de comparaison sauvegardé : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/model_comparison.csv
